In [1]:
!pip install xarray netCDF4
! pip install matplotlib seaborn

In [2]:
import sys
!{sys.executable} -m pip install scikit-learn

In [3]:
!{sys.executable} -m pip install -r requirements.txt

  Using cached accelerate-1.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached aiofiles-25.1.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiohttp-3.10.11-cp310-cp310-macosx_11_0_arm64.whl.metadata (7.7 kB)
  Using cached aiohttp_retry-2.8.3-py3-none-any.whl.metadata (8.9 kB)
  Using cached aioice-0.10.1-py3-none-any.whl.metadata (4.1 kB)
  Using cached aiortc-1.14.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached albucore-0.0.33-py3-none-any.whl.metadata (7.8 kB)
  Using cached albumentationsx-2.0.11-py3-none-any.whl.metadata (79 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached anthropic-0.49.0-py3-none-any.whl.metadata (24 kB)
  Using cached anyio-4.11.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached anywidget-0.9.18-py3-none-a

In [4]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score
)
import os
import imageio.v2 as imageio
from datetime import datetime, timedelta
from scipy import ndimage


## Cleaned ERA5 LAND dataset

In [ ]:
era5_land = xr.open_dataset("clean_datasets/era5_land_two_var_1990_2020.nc")
print(era5_land)

<xarray.Dataset> Size: 3GB
Dimensions:    (lat: 361, lon: 701, time: 2852)
Coordinates:
    number     int64 8B ...
  * lat        (lat) float64 3kB 35.0 35.1 35.2 35.3 ... 70.7 70.8 70.9 71.0
  * lon        (lon) float64 6kB -25.0 -24.9 -24.8 -24.7 ... 44.7 44.8 44.9 45.0
  * time       (time) datetime64[ns] 23kB 1990-06-01 1990-06-02 ... 2020-08-31
Data variables:
    swvl1      (time, lat, lon) float32 3GB ...
    land_mask  (lat, lon) bool 253kB ...


## Cleaned ERA 5 pressure dataset

In [ ]:
era5_pressure = xr.open_dataset("clean_datasets/era5_pressure_1990_2020_merged_clean.nc")
print(era5_pressure)

<xarray.Dataset> Size: 1GB
Dimensions:         (time: 2852, pressure_level: 1, lat: 145, lon: 281)
Coordinates:
    number          int64 8B ...
  * pressure_level  (pressure_level) float64 8B 500.0
  * lat             (lat) float64 1kB 35.0 35.25 35.5 35.75 ... 70.5 70.75 71.0
  * lon             (lon) float64 2kB -25.0 -24.75 -24.5 ... 44.5 44.75 45.0
  * time            (time) datetime64[ns] 23kB 1990-06-01 ... 2020-08-31
Data variables:
    v               (time, pressure_level, lat, lon) float32 465MB ...
    u               (time, pressure_level, lat, lon) float32 465MB ...
    z               (time, pressure_level, lat, lon) float32 465MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-08T16:45 GRIB to CDM+CF via cfgr

## Cleaned CN dataset 

In [ ]:
clean_ds = xr.open_dataset("clean_datasets/complex_network_dataset.nc")
print(clean_ds)

<xarray.Dataset> Size: 3GB
Dimensions:      (lon: 268, lat: 141, time: 2790)
Coordinates:
  * lon          (lon) float32 1kB -26.0 -25.75 -25.5 -25.25 ... 40.5 40.75 41.0
  * lat          (lat) float32 564B 35.8 36.05 36.3 36.55 ... 70.5 70.75 71.0
  * time         (time) datetime64[ns] 22kB 1990-06-01 1990-06-02 ... 2020-08-31
Data variables:
    DC           (time, lat, lon) float32 422MB ...
    CC           (time, lat, lon) float32 422MB ...
    BC           (time, lat, lon) float32 422MB ...
    ID           (time, lat, lon) float32 422MB ...
    OD           (time, lat, lon) float32 422MB ...
    is_heatwave  (time, lat, lon) float32 422MB ...


#### align times (knowing CN dataset is missing 1-31.08.1997 and 1-31.07.1999)

In [8]:

# CN time is your master time axis
common_time = clean_ds.time

# keep only dates present in CN
era5_land_time = era5_land.sel(time=common_time)
era5_pressure_time = era5_pressure.sel(time=common_time)

print(len(clean_ds.time), len(era5_land_time.time), len(era5_pressure_time.time))
print(np.array_equal(clean_ds.time.values, era5_land_time.time.values))
print(np.array_equal(clean_ds.time.values, era5_pressure_time.time.values))

2790 2790 2790
True
True


## clean ERA 5 pressure dataset

In [9]:
if "pressure_level" in era5_pressure_time.dims and era5_pressure_time.sizes["pressure_level"] == 1:
    era5_pressure_time = era5_pressure_time.squeeze("pressure_level", drop=True)

if "number" in era5_pressure_time.coords:
    era5_pressure_time = era5_pressure_time.drop_vars("number")

if "number" in era5_land_time.coords:
    era5_land_time = era5_land_time.drop_vars("number")

In [10]:
# Fix longitude: remove lon = -26 and start from -25
clean_ds = clean_ds.sel(lon=slice(-25.0, None))
era5_land = era5_land.sel(lon=slice(-25.0, None))
era5_pressure = era5_pressure.sel(lon=slice(-25.0, None))

In [11]:
print("CN lon min/max:", float(clean_ds.lon.min()), float(clean_ds.lon.max()))
print("Land lon min/max:", float(era5_land.lon.min()), float(era5_land.lon.max()))
print("Pressure lon min/max:", float(era5_pressure.lon.min()), float(era5_pressure.lon.max()))

# Add time dimension to land_mask if it only has (lat, lon)
if era5_land["land_mask"].dims == ("lat", "lon"):
    era5_land["land_mask"] = era5_land["land_mask"].expand_dims(time=era5_land.time)
    era5_land["land_mask"] = era5_land["land_mask"].transpose("time", "lat", "lon")

print("land_mask dims:", era5_land["land_mask"].dims)

CN lon min/max: -24.996253967285156 41.0
Land lon min/max: -25.0 45.0
Pressure lon min/max: -25.0 45.0
land_mask dims: ('time', 'lat', 'lon')


In [12]:
# clean_ds.to_netcdf("clean_cn_time_aligned.nc")
# era5_land_time.to_netcdf("era5_land_time_aligned_to_cn.nc")
# era5_pressure_time.to_netcdf("era5_pressure_time_aligned_to_cn.nc")

In [12]:
# 1. CN is the master dataset
cn_ds = clean_ds.sortby("time").sortby("lat").sortby("lon")

# 2. Restrict ERA5 to CN time and sort
era5_land_cn_time = era5_land.sel(time=cn_ds.time).sortby("time").sortby("lat").sortby("lon")
era5_pressure_cn_time = era5_pressure.sel(time=cn_ds.time).sortby("time").sortby("lat").sortby("lon")

# 3. Clean extra dimensions / coords
if "pressure_level" in era5_pressure_cn_time.dims and era5_pressure_cn_time.sizes["pressure_level"] == 1:
    era5_pressure_cn_time = era5_pressure_cn_time.squeeze("pressure_level", drop=True)

for coord in ["number", "expver", "surface", "valid_time"]:
    if coord in era5_land_cn_time.coords:
        era5_land_cn_time = era5_land_cn_time.drop_vars(coord)
    if coord in era5_pressure_cn_time.coords:
        era5_pressure_cn_time = era5_pressure_cn_time.drop_vars(coord)


In [13]:

# 4. Reindex ONLY lat/lon, keep time as already selected
swvl1_on_cn = era5_land_cn_time["swvl1"].reindex(
    lat=cn_ds.lat,
    lon=cn_ds.lon,
    method="nearest",
    tolerance=0.2
)

In [14]:
land_mask_on_cn = era5_land_cn_time["land_mask"].reindex(
    lat=cn_ds.lat,
    lon=cn_ds.lon,
    method="nearest",
    tolerance=0.2
)

In [15]:
u_on_cn = era5_pressure_cn_time["u"].reindex(
    lat=cn_ds.lat,
    lon=cn_ds.lon,
    method="nearest",
    tolerance=0.2
)

In [16]:
v_on_cn = era5_pressure_cn_time["v"].reindex(
    lat=cn_ds.lat,
    lon=cn_ds.lon,
    method="nearest",
    tolerance=0.2
)

In [17]:
z_on_cn = era5_pressure_cn_time["z"].reindex(
    lat=cn_ds.lat,
    lon=cn_ds.lon,
    method="nearest",
    tolerance=0.2
)


In [18]:
# 5. Build datasets directly from reindexed DataArrays
# no assign_coords(time=...) here
era5_land_on_cn = xr.merge([swvl1_on_cn.rename("swvl1"),
                            land_mask_on_cn.rename("land_mask")])

era5_pressure_on_cn = xr.merge([u_on_cn.rename("u"),
                                v_on_cn.rename("v"),
                                z_on_cn.rename("z")])

# 6. Optional: remove non-dimension coords before final merge
cn_ds_clean = cn_ds.reset_coords(drop=True)
era5_land_on_cn = era5_land_on_cn.reset_coords(drop=True)
era5_pressure_on_cn = era5_pressure_on_cn.reset_coords(drop=True)

# 7. Check alignment
print("Time match land:", np.array_equal(cn_ds_clean.time.values, era5_land_on_cn.time.values))
print("Time match pressure:", np.array_equal(cn_ds_clean.time.values, era5_pressure_on_cn.time.values))
print("Lat match land:", np.array_equal(cn_ds_clean.lat.values, era5_land_on_cn.lat.values))
print("Lat match pressure:", np.array_equal(cn_ds_clean.lat.values, era5_pressure_on_cn.lat.values))
print("Lon match land:", np.array_equal(cn_ds_clean.lon.values, era5_land_on_cn.lon.values))
print("Lon match pressure:", np.array_equal(cn_ds_clean.lon.values, era5_pressure_on_cn.lon.values))

Time match land: True
Time match pressure: True
Lat match land: True
Lat match pressure: True
Lon match land: True
Lon match pressure: True


In [19]:
# 8. Check CC before merge
print("CC before merge:",
      cn_ds_clean["CC"].mean().item(),
      cn_ds_clean["CC"].std().item(),
      cn_ds_clean["CC"].min().item(),
      cn_ds_clean["CC"].max().item())

# 9. Merge
merged_cn = xr.merge(
    [cn_ds_clean, era5_land_on_cn, era5_pressure_on_cn],
    compat="override",
    join="exact"
)

# 10. Check CC after merge
print("CC after merge:",
      merged_cn["CC"].mean().item(),
      merged_cn["CC"].std().item(),
      merged_cn["CC"].min().item(),
      merged_cn["CC"].max().item())

CC before merge: 0.057089876383543015 0.22476397454738617 0.0 1.0
CC after merge: 0.057089876383543015 0.22476397454738617 0.0 1.0


In [20]:
for var in merged_cn.data_vars:
    if merged_cn[var].dtype == "float64":
        merged_cn[var] = merged_cn[var].astype("float32")

In [ ]:
print(merged_cn)
# merged_cn = merged_cn.load()

<xarray.Dataset> Size: 4GB
Dimensions:      (lon: 264, lat: 141, time: 2790)
Coordinates:
  * lon          (lon) float32 1kB -25.0 -24.75 -24.49 ... 40.5 40.75 41.0
  * lat          (lat) float32 564B 35.8 36.05 36.3 36.55 ... 70.5 70.75 71.0
  * time         (time) datetime64[ns] 22kB 1990-06-01 1990-06-02 ... 2020-08-31
Data variables:
    DC           (time, lat, lon) float32 415MB ...
    CC           (time, lat, lon) float32 415MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    BC           (time, lat, lon) float32 415MB ...
    ID           (time, lat, lon) float32 415MB ...
    OD           (time, lat, lon) float32 415MB ...
    is_heatwave  (time, lat, lon) float32 415MB ...
    swvl1        (time, lat, lon) float32 415MB ...
    land_mask    (time, lat, lon) bool 104MB False False False ... False False
    u            (time, lat, lon) float32 415MB ...
    v            (time, lat, lon) float32 415MB ...
    z            (time, lat, lon) float32 415MB ...


In [ ]:
output_path = "/Users/malgorzatazdych/Documents/Master2/Thesis/Dataset/MasterThesis/merged_clean_cn_land_pressure_fixed.nc"
merged_cn.to_netcdf(output_path)
print("Saved to:", output_path)

Saved to: /Users/malgorzatazdych/Documents/Master2/Thesis/Dataset/MasterThesis/merged_clean_cn_land_pressure_NOW.nc


In [24]:
import xarray as xr

ds_check = xr.open_dataset(output_path)

print("Lon min/max:", float(ds_check.lon.min()), float(ds_check.lon.max()))
print("nlon:", ds_check.sizes["lon"])
print(ds_check)

Lon min/max: -24.996253967285156 41.0
nlon: 264
<xarray.Dataset> Size: 4GB
Dimensions:      (lon: 264, lat: 141, time: 2790)
Coordinates:
  * lon          (lon) float32 1kB -25.0 -24.75 -24.49 ... 40.5 40.75 41.0
  * lat          (lat) float32 564B 35.8 36.05 36.3 36.55 ... 70.5 70.75 71.0
  * time         (time) datetime64[ns] 22kB 1990-06-01 1990-06-02 ... 2020-08-31
Data variables:
    DC           (time, lat, lon) float32 415MB ...
    CC           (time, lat, lon) float32 415MB ...
    BC           (time, lat, lon) float32 415MB ...
    ID           (time, lat, lon) float32 415MB ...
    OD           (time, lat, lon) float32 415MB ...
    is_heatwave  (time, lat, lon) float32 415MB ...
    swvl1        (time, lat, lon) float32 415MB ...
    land_mask    (time, lat, lon) bool 104MB ...
    u            (time, lat, lon) float32 415MB ...
    v            (time, lat, lon) float32 415MB ...
    z            (time, lat, lon) float32 415MB ...


In [25]:
for var in ["swvl1", "u", "v", "z"]:
    print(var, bool(ds_check[var].isel(time=0).isnull().any()))

print("land_mask", bool(ds_check["land_mask"].isnull().any()))

swvl1 False
u False
v False
z False
land_mask False


In [26]:
for var in ["swvl1", "land_mask", "u", "v", "z"]:
    print(var, bool(ds_check[var].isnull().any()))

swvl1 False
land_mask False
u False
v False
z False


## Final file


In [ ]:
import xarray as xr
import pandas as pd

file_path = "/Users/malgorzatazdych/Documents/Master2/Thesis/Dataset/MasterThesis/merged_clean_cn_land_pressure_fixed.nc"
ds = xr.open_dataset(file_path)

print("=== BASIC SHAPE ===")
print(ds)

print("\n=== COORDINATE RANGES ===")
print("lat min/max:", float(ds.lat.min()), float(ds.lat.max()), "nlat:", ds.sizes["lat"])
print("lon min/max:", float(ds.lon.min()), float(ds.lon.max()), "nlon:", ds.sizes["lon"])
print("time min/max:", pd.to_datetime(ds.time.values[0]), "->", pd.to_datetime(ds.time.values[-1]), "ntime:", ds.sizes["time"])

print("\n=== TIME COUNTS ===")
t = pd.DatetimeIndex(pd.to_datetime(ds.time.values))
print("Total timesteps:", len(t))
print("Unique timesteps:", len(pd.unique(t)))
print("Any duplicated times:", len(t) != len(pd.unique(t)))

print("\n=== YEAR COUNTS ===")
year_counts = pd.Series(t.year).value_counts().sort_index()
print(year_counts)

print("\n=== MISSING DATES VS FULL JJA 1990-2020 ===")
expected = pd.DatetimeIndex(
    [d for year in range(1990, 2021)
     for d in pd.date_range(f"{year}-06-01", f"{year}-08-31", freq="D")]
)
missing = expected.difference(t)
print("Missing dates:", len(missing))
print(missing.tolist())

print("\n=== NaNs PER VARIABLE ===")
for var in ds.data_vars:
    try:
        print(f"{var}: {int(ds[var].isnull().sum())}")
    except Exception as e:
        print(f"{var}: error while checking NaNs -> {e}")

ds.close()

=== BASIC SHAPE ===
<xarray.Dataset> Size: 4GB
Dimensions:      (lon: 264, lat: 141, time: 2790)
Coordinates:
  * lon          (lon) float32 1kB -25.0 -24.75 -24.49 ... 40.5 40.75 41.0
  * lat          (lat) float32 564B 35.8 36.05 36.3 36.55 ... 70.5 70.75 71.0
  * time         (time) datetime64[ns] 22kB 1990-06-01 1990-06-02 ... 2020-08-31
Data variables:
    DC           (time, lat, lon) float32 415MB ...
    CC           (time, lat, lon) float32 415MB ...
    BC           (time, lat, lon) float32 415MB ...
    ID           (time, lat, lon) float32 415MB ...
    OD           (time, lat, lon) float32 415MB ...
    is_heatwave  (time, lat, lon) float32 415MB ...
    swvl1        (time, lat, lon) float32 415MB ...
    land_mask    (time, lat, lon) bool 104MB ...
    u            (time, lat, lon) float32 415MB ...
    v            (time, lat, lon) float32 415MB ...
    z            (time, lat, lon) float32 415MB ...

=== COORDINATE RANGES ===
lat min/max: 35.79999923706055 71.0 nlat: 14

In [28]:
print(ds["CC"])
print("CC mean:", float(ds["CC"].mean()))
print("CC std:", float(ds["CC"].std()))
print("CC min:", float(ds["CC"].min()))
print("CC max:", float(ds["CC"].max()))

<xarray.DataArray 'CC' (time: 2790, lat: 141, lon: 264)> Size: 415MB
array([[[0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ],
        ...,
        [0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ]],

       [[0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.993267, 0.      ],
        ...,
        [0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ]],

       ...,

       [[0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ],
        ...,
        [0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ]],

       [[0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ],
        ...,
        [0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0

In [29]:
for var in ds.data_vars:
    print(f"\n--- {var} ---")
    print("dims:", ds[var].dims)
    print("shape:", ds[var].shape)
    print("dtype:", ds[var].dtype)
    print("NaNs:", int(ds[var].isnull().sum()))
    print("mean:", float(ds[var].mean()))
    print("std:", float(ds[var].std()))
    print("min:", float(ds[var].min()))
    print("max:", float(ds[var].max()))


--- DC ---
dims: ('time', 'lat', 'lon')
shape: (2790, 141, 264)
dtype: float32
NaNs: 0
mean: 0.0051677031442523
std: 0.0229827631264925
min: 0.0
max: 0.19099199771881104

--- CC ---
dims: ('time', 'lat', 'lon')
shape: (2790, 141, 264)
dtype: float32
NaNs: 0
mean: 0.057089876383543015
std: 0.22476397454738617
min: 0.0
max: 1.0

--- BC ---
dims: ('time', 'lat', 'lon')
shape: (2790, 141, 264)
dtype: float32
NaNs: 0
mean: 2.9350235308811534e-07
std: 1.2729590253002243e-06
min: 0.0
max: 9.999999747378752e-06

--- ID ---
dims: ('time', 'lat', 'lon')
shape: (2790, 141, 264)
dtype: float32
NaNs: 0
mean: 41.32040023803711
std: 286.3206787109375
min: 0.0
max: 4493.0

--- OD ---
dims: ('time', 'lat', 'lon')
shape: (2790, 141, 264)
dtype: float32
NaNs: 0
mean: 41.184288024902344
std: 283.764892578125
min: 0.0
max: 4460.0

--- is_heatwave ---
dims: ('time', 'lat', 'lon')
shape: (2790, 141, 264)
dtype: float32
NaNs: 0
mean: 0.04983643442392349
std: 0.2172907143831253
min: 0.0
max: 1.0

--- swvl1 --

## EXTRA Check NaNs - runs for a very long time for era5_pressure

In [ ]:
for var in era5_land_on_cn.data_vars:
    print(f"{var} NaNs:", int(era5_land_on_cn[var].isnull().sum()))

for var in era5_pressure_on_cn.data_vars:
    print(f"{var} NaNs:", int(era5_pressure_on_cn[var].isnull().sum()))

swvl1 NaNs: 0
land_mask NaNs: 0


In [ ]:
merged_cn = xr.merge([cn_ds, era5_land_on_cn, era5_pressure_on_cn])
print(merged_cn)

In [ ]:
output_path = "/Users/malgorzatazdych/Documents/Master2/Thesis/Dataset/MasterThesis/merged_FINAL.nc"
merged_cn.to_netcdf(output_path)
print("Saved to:", output_path)

Degree Centrality (DC)

How synchronized a region is with others.

➡ high = many regions experiencing heat together
➡ hubs of heatwave activity

Clustering Coefficient (CC)

Local clustering of heatwaves.

➡ high = localized heatwave cluster
➡ stationary behaviour

Betweenness Centrality (BC)

Propagation corridor importance.

➡ high = heatwave movement pathway
➡ dynamic propagation routes

Use CN metrics to:

✔ identify propagation corridors
✔ detect dynamic heatwaves
✔ validate DL trajectories

Use is_heatwave to:

✔ define events
✔ train models
✔ detect movement

Use E-OBS Tmax to:

✔ build sequences
✔ provide physical context